In [2]:
import os
import tempfile
import PIL.Image
import torch
from transformers import AutoModel, AutoTokenizer


import PIL.Image
import os
import tempfile
import torch

from transformers import AutoImageProcessor
from transformers import AutoModel, AutoTokenizer
from transformers.models.detr import DetrForSegmentation
from doclayout_yolo import YOLOv10

from src.extract_data import ExtractData

In [3]:
import cv2
from doclayout_yolo import YOLOv10

# Load the pre-trained model
model = YOLOv10("models/doclayout_yolo_docstructbench_imgsz1024.pt")

# Perform prediction
det_res = model.predict(
    "/home/proven/vlm_data_extraction/data/image/Cansu-Gökhan-Şubat 2025.jpeg",   # Image to predict
    imgsz=1024,        # Prediction image size
    conf=0.1,          # Confidence threshold
    device="cuda:0"    # Device to use (e.g., 'cuda:0' or 'cpu')
)

# Annotate and save the result
annotated_frame = det_res[0].plot(pil=True, line_width=5, font_size=20)
cv2.imwrite("result.jpg", annotated_frame)


image 1/1 /home/proven/vlm_data_extraction/data/image/Cansu-Gökhan-Şubat 2025.jpeg: 1024x768 1 title, 4 plain texts, 5 abandons, 2 figures, 3 tables, 1 table_footnote, 40.8ms
Speed: 3.9ms preprocess, 40.8ms inference, 46.7ms postprocess per image at shape (1, 3, 1024, 768)


True

In [4]:
from src.extract_data import ExtractData
from doclayout_yolo import YOLOv10


class ExtractDataFromImageYOLO(ExtractData):
    def __init__(self):
        super().__init__()
        self.model = YOLOv10(
            "models/doclayout_yolo_docstructbench_imgsz1024.pt")
        self.device = "cuda:0"
        self.conf = 0.1
        self.imgsize = 1024

        self.tokenizer = AutoTokenizer.from_pretrained(
            'ucaslcl/GOT-OCR2_0', trust_remote_code=True)
        self.ocr_model = AutoModel.from_pretrained('ucaslcl/GOT-OCR2_0', trust_remote_code=True, low_cpu_mem_usage=True,
                                                   device_map='cuda', use_safetensors=True, pad_token_id=self.tokenizer.eos_token_id)
        self.ocr_model = self.ocr_model.eval().cuda()

    def _get_yolo_attr(self, boxes):
        bboxes = boxes.xyxy.cpu().numpy()
        classes = boxes.cls.cpu().numpy()
        class_names = self.model.names if hasattr(
            self.model, 'names') else None

        _bboxes = []
        _labels = []
        for box in range(len(bboxes)):
            x1, y1, x2, y2 = bboxes[box]
            class_idx = int(classes[box])
            label = class_names[class_idx] if class_names else f"Class {class_idx}"

            _bboxes.append([int(x1), int(y1), int(x2), int(y2)])
            _labels.append(label)

        return _bboxes, _labels

    def _get_bbox_with_yolo(self, image_path):
        results = self.model.predict(
            image_path,
            imgsz= self.imgsize,
            conf=self.conf,
            device= self.device
        )
        boxes = results[0].boxes
        bboxes, labels = self._get_yolo_attr(boxes)
        return bboxes, labels

    def _crop_yolo_image(self, image_path: str, bboxes, labels) -> list:
        cropped_images = []
        image = PIL.Image.open(image_path)
        for bbox, label in zip(bboxes, labels):
            if label not in []:
                xmin, ymin, xmax, ymax = bbox
                cropped_image = image.crop((xmin -5, ymin -5, xmax+ 5, ymax+ 5))
                cropped_images.append(cropped_image)
        return cropped_images

    def extract_text_with_OCR(self, image_path: str):
        total_string = ""
        bboxes, labels = self._get_bbox_with_yolo(image_path)
        cropped_images = self._crop_yolo_image(image_path, bboxes, labels)

        with tempfile.TemporaryDirectory() as temp_dir:
            # Save each image temporarily
            for i, piece in enumerate(cropped_images):
                temp_path = os.path.join(temp_dir, f"temp_image_{i}.png")
                piece.save(temp_path)
                res = self.ocr_model.chat(
                    self.tokenizer, temp_path, ocr_type='ocr')

                total_string += f"Text Cluster_{i}: {res}" + "\n\n"
        return total_string

In [11]:
extract_object = ExtractDataFromImageYOLO()
img= "/home/proven/vlm_data_extraction/data/image/Cansu-Gökhan-Şubat 2025.jpeg"

def extract_image(object: ExtractDataFromImageYOLO, image_file):
        text = object.extract_text_with_OCR(image_file)
        information = object.get_information(text)

        return information, text 

inf, text= extract_image(extract_object, img)



image 1/1 /home/proven/vlm_data_extraction/data/image/Cansu-Gökhan-Şubat 2025.jpeg: 1024x768 1 title, 4 plain texts, 5 abandons, 2 figures, 3 tables, 1 table_footnote, 12.1ms
Speed: 2.1ms preprocess, 12.1ms inference, 0.4ms postprocess per image at shape (1, 3, 1024, 768)


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end gene

In [9]:
print(inf)

{
  "totalAmount": "5,000.00 TL",
  "invoiceNumber": "FS02025000000245",
  "sellerRegistrationNumber": "63577401700"
}


In [12]:
print(text)

Text Cluster_0: BelgeNumaras: FS02025000000245 DuzenlenmeTarihi: 2025-02-27 DuzenlenmeSaati: 19:26:01

Text Cluster_1: SraN Cerein No Gain Almango Dari Cerei G. V. Stopaj Orani % Net Cerei KDV Orani % KDV Textifat Orani % Net Tashita 0.00 4,545,45TL 10.00 5.000,00TL

Text Cluster_2: DUZEN LE YEN HALIL GOKH AN DEMIR KIRAN Cankaya/ 06 I ANKA RAT UR KIYE Tel: 0(312)4677440 Fax:  WebSites i: www. go khan demi rk iran. com E- Post a: dr go khan demi rk iran@ gmail. com Mers is No:  T CK N: 63577401700

Text Cluster_3: ACI K LAMA: YAL NIZ: BE SB IN TL

Text Cluster_4: 口 口

Text Cluster_5: BrutUcret: 4.545,45TL (*)G.V.StopajTutarn: 0,00TL NetUcretTutarn: 4.545,45TL KDVTutarn: 454,55TL (**)KDVTevkifatTutarn: 0,00TL TahsilEdilenKDVTutarn: 454,55TL NetAlinanToplam: 5.000,00TL

Text Cluster_6: * ) G. V. Stop a j Tut an: Bu tut aral ic it a raf in dan Muh t asar Bey an name ile beyan edi lip oden mes i gere ken tut ar dur.

Text Cluster_7: G i

Text Cluster_8: ALICIBILGILERI

Text Cluster_9: 口 口



In [1]:
import time

timestamp = int(time.time() * 1000)
print(timestamp)


1745931244364


In [ ]:
from src.extract_data_image import ExtractDataFromImageYOLO

ExtractDataFromImageYOLO